# Importing libraries and defining helper functions

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm.notebook import tqdm


# Step 1: Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")  # Use the first CUDA device
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")  # Fallback to CPU if GPU is not available
    print("GPU is not available, using CPU instead.")


# function to "corrupt" memory vectors, for use as initial condition to system
def generate_corruption_vector(length, corruption,device):
    """
    Generates a vector of 1s and -1s based on a specified corruption probability.

    Parameters:
    - length: The length of the output vector.
    - corruption: The probability of an element being -1 (p(-1)).

    Returns:
    - A torch.Tensor of shape (length,) with elements set to 1 with probability `1 - corruption` and to -1 with probability `corruption`.
    """
    # Generate a random vector with values between 0 and 1
    random_vector = torch.rand(length,device = device)

    # Elements <= 1 - corruption are set to 1, others are set to -1
    # This inverts the condition to meet the specified probabilities
    output_vector = torch.where(random_vector <= (1 - corruption), torch.tensor(1), torch.tensor(-1))

    return output_vector


Using GPU: NVIDIA GeForce GTX 1080 Ti


# Forward simulation of the neuron-astrocyte associative memory network

In [2]:
# Parameters
N = 500  # Number of neurons
K = 10  # Number of memories
NUM_STEPS = 1000  # Number of Euler steps for integration
dt = 0.05  # Euler step size
beta = 5  # Gain of nonlinearity
corruption_level = 0.1  # Controls the degree of corruption of the initial state

# Generate random binary memories
etas = 2 * torch.randint(2, (N, K), device=device).float() - 1

# Select which memory to initialize dynamics near
memory_to_seed_from = 2
mem = etas[:, memory_to_seed_from]

# Initialize all dynamic variables near the desired fixed point
x0 = mem * generate_corruption_vector(N, corruption_level, device)

# Initial states calculations
psi_0 = -torch.outer(torch.tanh(beta * x0), torch.tanh(beta * x0))
P0 = (1 / beta) * torch.atanh(psi_0)
S0 = (1 / beta) * torch.atanh(-(1 / N ** 3) * torch.einsum('im,jm,km,lm,kl->ij', etas, etas, etas, etas, psi_0))

# Initializing variables for dynamics
x, S, P = x0.clone(), S0.clone(), P0.clone()

# Euler integration of dynamics
for _ in tqdm(range(NUM_STEPS)):
    h = torch.tanh(beta * x)
    g = torch.tanh(beta * S)
    psi = torch.tanh(beta * P)

    x += dt * (-x + g @ h)
    S += dt * (-S + torch.outer(h, h) + psi)
    P += dt * (-P + (1 / N ** 3) * torch.einsum('im,jm,km,lm,kl->ij', etas, etas, etas, etas, psi) + g)

# Check whether dynamics converged to the correct attractor
initial_distance = torch.norm(torch.sign(x0) - mem)
final_distance = torch.norm(torch.sign(h) - mem)

print(f'Initial L2 distance of neural state to memory {memory_to_seed_from} is {initial_distance}')
print(f'Final L2 distance of neural state to memory {memory_to_seed_from} is {final_distance}')


  0%|          | 0/1000 [00:00<?, ?it/s]

Initial L2 distance of neural state to memory 2 is 14.966629981994629
Final L2 distance of neural state to memory 2 is 0.0
